In [0]:
from pyspark.sql import functions as F

# =========================================================
# CONFIGURATION
# =========================================================

BRONZE_PATH = "/Volumes/workspace/finance_analytics/finance_raw/bronze"
SILVER_PATH = "/Volumes/workspace/finance_analytics/finance_raw/silver"

# =========================================================
# LOAD BRONZE CUSTOMERS
# =========================================================

customers_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/customers")
)

print(f"Bronze customers: {customers_bronze_df.count():,}")

In [0]:
# =========================================================
# INSPECT BRONZE CUSTOMER SCHEMA
# =========================================================

customers_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/customers")
)

customers_bronze_df.printSchema()

display(customers_bronze_df.limit(10))

In [0]:
# =========================================================
# CLEAN CUSTOMERS
# =========================================================

customers_silver_df = (
    customers_bronze_df

    # Remove duplicate customer records
    .dropDuplicates(["customer_id"])

    # Standardise text fields
    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )
    .withColumn(
        "country",
        F.upper(F.trim(F.col("country")))
    )
    .withColumn(
        "customer_segment",
        F.initcap(F.trim(F.col("customer_segment")))
    )

    # Ensure correct data types
    .withColumn(
        "credit_score",
        F.col("credit_score").cast("int")
    )
    .withColumn(
        "customer_since",
        F.to_date("customer_since")
    )
)

display(customers_silver_df.limit(10))

In [0]:
# =========================================================
# CUSTOMER SILVER VALIDATION
# =========================================================

print(
    f"Silver customers before write: "
    f"{customers_silver_df.count():,}"
)

duplicate_customer_ids = (
    customers_silver_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate customer IDs: {duplicate_customer_ids}")

null_customer_ids = (
    customers_silver_df
    .filter(F.col("customer_id").isNull())
    .count()
)

print(f"Null customer IDs: {null_customer_ids}")

print("Credit score range:")

display(
    customers_silver_df.select(
        F.min("credit_score").alias("min_credit_score"),
        F.max("credit_score").alias("max_credit_score")
    )
)

In [0]:
# =========================================================
# WRITE CUSTOMERS TO SILVER AS DELTA
# =========================================================

customers_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/customers")

print("Customers successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER CUSTOMERS
# =========================================================

silver_customers_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/customers")
)

print(f"Silver customers: {silver_customers_check.count():,}")

display(silver_customers_check.limit(10))

In [0]:
# =========================================================
# LOAD BRONZE PRODUCTS
# =========================================================

products_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/products")
)

print(f"Bronze products: {products_bronze_df.count():,}")

products_bronze_df.printSchema()
display(products_bronze_df)

In [0]:
# =========================================================
# CLEAN PRODUCTS
# =========================================================

products_silver_df = (
    products_bronze_df

    # Remove duplicate products
    .dropDuplicates(["product_id"])

    # Standardise text fields
    .withColumn(
        "product_name",
        F.initcap(F.trim(F.col("product_name")))
    )
    .withColumn(
        "product_type",
        F.initcap(F.trim(F.col("product_type")))
    )

    # Ensure numeric data types
    .withColumn(
        "interest_rate",
        F.col("interest_rate").cast("double")
    )
    .withColumn(
        "annual_fee",
        F.col("annual_fee").cast("double")
    )
)

display(products_silver_df)

In [0]:
# =========================================================
# PRODUCT SILVER VALIDATION
# =========================================================

print(
    f"Silver products: "
    f"{products_silver_df.count():,}"
)

duplicate_products = (
    products_silver_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate product IDs: {duplicate_products}")

null_product_ids = (
    products_silver_df
    .filter(F.col("product_id").isNull())
    .count()
)

print(f"Null product IDs: {null_product_ids}")

print("Interest rate range:")

display(
    products_silver_df.select(
        F.min("interest_rate").alias("min_interest_rate"),
        F.max("interest_rate").alias("max_interest_rate")
    )
)

In [0]:
# =========================================================
# WRITE PRODUCTS TO SILVER AS DELTA
# =========================================================

products_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/products")

print("Products successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER PRODUCTS
# =========================================================

silver_products_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/products")
)

print(f"Silver products: {silver_products_check.count():,}")

display(silver_products_check)

In [0]:
# =========================================================
# LOAD BRONZE BRANCHES
# =========================================================

branches_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/branches")
)

print(f"Bronze branches: {branches_bronze_df.count():,}")

branches_bronze_df.printSchema()

display(branches_bronze_df)

In [0]:
# =========================================================
# CLEAN BRANCHES
# =========================================================

branches_silver_df = (
    branches_bronze_df

    # Remove duplicate branch records
    .dropDuplicates(["branch_id"])

    # Standardise text fields
    .withColumn(
        "branch_name",
        F.initcap(F.trim(F.col("branch_name")))
    )
    .withColumn(
        "city",
        F.initcap(F.trim(F.col("city")))
    )
    .withColumn(
        "region",
        F.initcap(F.trim(F.col("region")))
    )
    .withColumn(
        "country",
        F.upper(F.trim(F.col("country")))
    )
)

display(branches_silver_df)

In [0]:
# =========================================================
# BRANCH SILVER VALIDATION
# =========================================================

print(
    f"Silver branches: "
    f"{branches_silver_df.count():,}"
)

duplicate_branch_ids = (
    branches_silver_df
    .groupBy("branch_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate branch IDs: {duplicate_branch_ids}")

null_branch_ids = (
    branches_silver_df
    .filter(F.col("branch_id").isNull())
    .count()
)

print(f"Null branch IDs: {null_branch_ids}")

In [0]:
# =========================================================
# WRITE BRANCHES TO SILVER AS DELTA
# =========================================================

branches_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/branches")

print("Branches successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER BRANCHES
# =========================================================

silver_branches_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/branches")
)

print(f"Silver branches: {silver_branches_check.count():,}")

display(silver_branches_check)

In [0]:
# =========================================================
# LOAD BRONZE ACCOUNTS
# =========================================================

accounts_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/accounts")
)

print(f"Bronze accounts: {accounts_bronze_df.count():,}")

accounts_bronze_df.printSchema()

display(accounts_bronze_df.limit(10))

In [0]:
# =========================================================
# CLEAN ACCOUNTS
# =========================================================

accounts_silver_df = (
    accounts_bronze_df

    # Remove duplicate account records
    .dropDuplicates(["account_id"])

    # Standardise account status
    .withColumn(
        "account_status",
        F.initcap(F.trim(F.col("account_status")))
    )

    # Ensure correct data types
    .withColumn(
        "account_id",
        F.col("account_id").cast("long")
    )
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("int")
    )
    .withColumn(
        "product_id",
        F.col("product_id").cast("int")
    )
    .withColumn(
        "branch_id",
        F.col("branch_id").cast("int")
    )
    .withColumn(
        "open_date",
        F.to_date("open_date")
    )
    .withColumn(
        "current_balance",
        F.col("current_balance").cast("double")
    )
)

display(accounts_silver_df.limit(10))

In [0]:
# =========================================================
# ACCOUNT SILVER VALIDATION
# =========================================================

print(
    f"Silver accounts: "
    f"{accounts_silver_df.count():,}"
)

duplicate_account_ids = (
    accounts_silver_df
    .groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate account IDs: {duplicate_account_ids}")

null_account_ids = (
    accounts_silver_df
    .filter(F.col("account_id").isNull())
    .count()
)

print(f"Null account IDs: {null_account_ids}")

print("Account status distribution:")

display(
    accounts_silver_df
    .groupBy("account_status")
    .count()
    .orderBy("account_status")
)

print("Balance range:")

display(
    accounts_silver_df.select(
        F.min("current_balance").alias("min_balance"),
        F.max("current_balance").alias("max_balance")
    )
)

In [0]:
# =========================================================
# WRITE ACCOUNTS TO SILVER AS DELTA
# =========================================================

accounts_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/accounts")

print("Accounts successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER ACCOUNTS
# =========================================================

silver_accounts_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/accounts")
)

print(f"Silver accounts: {silver_accounts_check.count():,}")

display(silver_accounts_check.limit(10))

In [0]:
# =========================================================
# LOAD BRONZE TRANSACTIONS
# =========================================================

transactions_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/transactions")
)

print(f"Bronze transactions: {transactions_bronze_df.count():,}")

transactions_bronze_df.printSchema()

display(transactions_bronze_df.limit(10))

In [0]:
# =========================================================
# CLEAN TRANSACTIONS
# =========================================================

transactions_silver_df = (
    transactions_bronze_df

    # Remove duplicate transactions
    .dropDuplicates(["transaction_id"])

    # Standardise text fields
    .withColumn(
        "transaction_type",
        F.initcap(F.trim(F.col("transaction_type")))
    )
    .withColumn(
        "merchant_category",
        F.initcap(F.trim(F.col("merchant_category")))
    )
    .withColumn(
        "channel",
        F.initcap(F.trim(F.col("channel")))
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )

    # Ensure correct data types
    .withColumn(
        "transaction_id",
        F.col("transaction_id").cast("long")
    )
    .withColumn(
        "account_id",
        F.col("account_id").cast("int")
    )
    .withColumn(
        "transaction_date",
        F.to_date("transaction_date")
    )
    .withColumn(
        "amount",
        F.col("amount").cast("double")
    )
)

display(transactions_silver_df.limit(10))

In [0]:
# =========================================================
# TRANSACTION SILVER VALIDATION
# =========================================================

print(
    f"Silver transactions: "
    f"{transactions_silver_df.count():,}"
)

duplicate_transaction_ids = (
    transactions_silver_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Duplicate transaction IDs: "
    f"{duplicate_transaction_ids}"
)

null_transaction_ids = (
    transactions_silver_df
    .filter(F.col("transaction_id").isNull())
    .count()
)

print(
    f"Null transaction IDs: "
    f"{null_transaction_ids}"
)

negative_amounts = (
    transactions_silver_df
    .filter(F.col("amount") < 0)
    .count()
)

print(
    f"Negative transaction amounts: "
    f"{negative_amounts}"
)

print("Transaction type distribution:")

display(
    transactions_silver_df
    .groupBy("transaction_type")
    .count()
    .orderBy("transaction_type")
)

print("Transaction date range:")

display(
    transactions_silver_df.select(
        F.min("transaction_date").alias("min_date"),
        F.max("transaction_date").alias("max_date")
    )
)

In [0]:
# =========================================================
# WRITE TRANSACTIONS TO SILVER AS DELTA
# =========================================================

transactions_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/transactions")

print("Transactions successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER TRANSACTIONS
# =========================================================

silver_transactions_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/transactions")
)

print(
    f"Silver transactions: "
    f"{silver_transactions_check.count():,}"
)

display(silver_transactions_check.limit(10))

In [0]:
# =========================================================
# LOAD BRONZE LOANS
# =========================================================

loans_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/loans")
)

print(f"Bronze loans: {loans_bronze_df.count():,}")

loans_bronze_df.printSchema()

display(loans_bronze_df.limit(10))

In [0]:
# =========================================================
# CLEAN LOANS
# =========================================================

loans_silver_df = (
    loans_bronze_df

    # Remove duplicate loan records
    .dropDuplicates(["loan_id"])

    # Standardise loan status
    .withColumn(
        "loan_status",
        F.initcap(F.trim(F.col("loan_status")))
    )

    # Ensure correct data types
    .withColumn(
        "loan_id",
        F.col("loan_id").cast("long")
    )
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("int")
    )
    .withColumn(
        "product_id",
        F.col("product_id").cast("int")
    )
    .withColumn(
        "branch_id",
        F.col("branch_id").cast("int")
    )
    .withColumn(
        "loan_amount",
        F.col("loan_amount").cast("double")
    )
    .withColumn(
        "outstanding_balance",
        F.col("outstanding_balance").cast("double")
    )
    .withColumn(
        "interest_rate",
        F.col("interest_rate").cast("double")
    )
    .withColumn(
        "start_date",
        F.to_date("start_date")
    )
    .withColumn(
        "maturity_date",
        F.to_date("maturity_date")
    )
    .withColumn(
        "credit_score_at_origination",
        F.col("credit_score_at_origination").cast("int")
    )
)

display(loans_silver_df.limit(10))

In [0]:
# =========================================================
# LOAN SILVER VALIDATION
# =========================================================

print(
    f"Silver loans: "
    f"{loans_silver_df.count():,}"
)

duplicate_loan_ids = (
    loans_silver_df
    .groupBy("loan_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate loan IDs: {duplicate_loan_ids}")

null_loan_ids = (
    loans_silver_df
    .filter(F.col("loan_id").isNull())
    .count()
)

print(f"Null loan IDs: {null_loan_ids}")

negative_loan_amounts = (
    loans_silver_df
    .filter(F.col("loan_amount") < 0)
    .count()
)

print(f"Negative loan amounts: {negative_loan_amounts}")

invalid_balances = (
    loans_silver_df
    .filter(
        (F.col("outstanding_balance") < 0) |
        (F.col("outstanding_balance") > F.col("loan_amount"))
    )
    .count()
)

print(f"Invalid outstanding balances: {invalid_balances}")

invalid_credit_scores = (
    loans_silver_df
    .filter(
        (F.col("credit_score_at_origination") < 300) |
        (F.col("credit_score_at_origination") > 850)
    )
    .count()
)

print(
    f"Invalid credit scores: "
    f"{invalid_credit_scores}"
)

print("Loan status distribution:")

display(
    loans_silver_df
    .groupBy("loan_status")
    .count()
    .orderBy("loan_status")
)

In [0]:
# =========================================================
# WRITE LOANS TO SILVER AS DELTA
# =========================================================

loans_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/loans")

print("Loans successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER LOANS
# =========================================================

silver_loans_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/loans")
)

print(f"Silver loans: {silver_loans_check.count():,}")

display(silver_loans_check.limit(10))

In [0]:
# =========================================================
# LOAD BRONZE LOAN PAYMENTS
# =========================================================

loan_payments_bronze_df = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/loan_payments")
)

print(
    f"Bronze loan payments: "
    f"{loan_payments_bronze_df.count():,}"
)

loan_payments_bronze_df.printSchema()

display(loan_payments_bronze_df.limit(10))

In [0]:
# =========================================================
# CLEAN LOAN PAYMENTS
# =========================================================

loan_payments_silver_df = (
    loan_payments_bronze_df

    # Remove duplicate payment records
    .dropDuplicates(["payment_id"])

    # Standardise payment status
    .withColumn(
        "payment_status",
        F.initcap(F.trim(F.col("payment_status")))
    )

    # Ensure correct data types
    .withColumn(
        "payment_id",
        F.col("payment_id").cast("long")
    )
    .withColumn(
        "loan_id",
        F.col("loan_id").cast("int")
    )
    .withColumn(
        "payment_date",
        F.to_date("payment_date")
    )
    .withColumn(
        "payment_amount",
        F.col("payment_amount").cast("double")
    )
    .withColumn(
        "principal_amount",
        F.col("principal_amount").cast("double")
    )
    .withColumn(
        "interest_amount",
        F.col("interest_amount").cast("double")
    )
    .withColumn(
        "days_past_due",
        F.col("days_past_due").cast("int")
    )
)

display(loan_payments_silver_df.limit(10))

In [0]:
# =========================================================
# LOAN PAYMENT SILVER VALIDATION
# =========================================================

print(
    f"Silver loan payments: "
    f"{loan_payments_silver_df.count():,}"
)

duplicate_payment_ids = (
    loan_payments_silver_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Duplicate payment IDs: "
    f"{duplicate_payment_ids}"
)

null_payment_ids = (
    loan_payments_silver_df
    .filter(F.col("payment_id").isNull())
    .count()
)

print(
    f"Null payment IDs: "
    f"{null_payment_ids}"
)

negative_payment_amounts = (
    loan_payments_silver_df
    .filter(F.col("payment_amount") < 0)
    .count()
)

print(
    f"Negative payment amounts: "
    f"{negative_payment_amounts}"
)

negative_days_past_due = (
    loan_payments_silver_df
    .filter(F.col("days_past_due") < 0)
    .count()
)

print(
    f"Negative days past due: "
    f"{negative_days_past_due}"
)

payment_calculation_errors = (
    loan_payments_silver_df
    .filter(
        F.abs(
            F.col("payment_amount")
            - F.col("principal_amount")
            - F.col("interest_amount")
        ) > 0.01
    )
    .count()
)

print(
    f"Payment calculation errors: "
    f"{payment_calculation_errors}"
)

print("Payment status distribution:")

display(
    loan_payments_silver_df
    .groupBy("payment_status")
    .count()
    .orderBy("payment_status")
)

In [0]:
# =========================================================
# WRITE LOAN PAYMENTS TO SILVER AS DELTA
# =========================================================

loan_payments_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/loan_payments")

print("Loan payments successfully written to Silver.")

In [0]:
# =========================================================
# VERIFY SILVER LOAN PAYMENTS
# =========================================================

silver_loan_payments_check = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/loan_payments")
)

print(
    f"Silver loan payments: "
    f"{silver_loan_payments_check.count():,}"
)

display(silver_loan_payments_check.limit(10))

In [0]:
# =========================================================
# FINAL SILVER LAYER VALIDATION
# =========================================================

silver_datasets = [
    "customers",
    "products",
    "branches",
    "accounts",
    "transactions",
    "loans",
    "loan_payments"
]

print("=== SILVER LAYER VALIDATION ===")

for dataset in silver_datasets:

    df = (
        spark.read
        .format("delta")
        .load(f"{SILVER_PATH}/{dataset}")
    )

    print(f"{dataset}: {df.count():,} rows")

print("\nSilver layer validation completed successfully.")